# Coffee17 preprocessing — Kaggle final analysis

Tambahkan Coffee17 dataset dan output notebook OOF sebagai Kaggle Inputs. Notebook ini tidak melakukan training.


In [ ]:
CODE_COMMIT='7de2abb46efd5e71dbab508e2ae4b4e61102a7aa'
import hashlib, importlib, os, shutil, subprocess, sys, zipfile
from pathlib import Path
INPUT=Path('/kaggle/input'); WORK=Path('/kaggle/working')
PROJECT=WORK/'coffee17-preprocessing-project'; REPO=WORK/'coffee-bean-classification-code'

def sha256_file(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda:f.read(1024*1024), b''): h.update(block)
    return h.hexdigest()
def merge_tree_exact(source,target):
    for item in sorted(Path(source).rglob('*')):
        if not item.is_file(): continue
        dst=Path(target)/item.relative_to(source); dst.parent.mkdir(parents=True,exist_ok=True)
        if dst.is_file() and sha256_file(item)!=sha256_file(dst):
            raise RuntimeError(f'Input conflict: {item.relative_to(source)}')
        if not dst.exists(): shutil.copy2(item,dst)

prior=sorted(p for p in INPUT.rglob('coffee17-preprocessing-project') if p.is_dir())
if not prior: raise FileNotFoundError('Tambahkan output OOF sebagai Kaggle Input.')
for p in prior: merge_tree_exact(p,PROJECT)

if REPO.exists(): shutil.rmtree(REPO)
subprocess.run(['git','clone','--quiet','--no-checkout','https://github.com/ediprin/coffee-bean-classification.git',str(REPO)],check=True)
subprocess.run(['git','-C',str(REPO),'checkout','--quiet','--detach',CODE_COMMIT],check=True)
lock=PROJECT/'evidence/coffee17-preprocessing-runtime-v1/requirements_preprocessing_study_lock.txt'
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(lock)],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps','-e',str(REPO)],check=True)
sys.path.insert(0,str(REPO/'src')); importlib.invalidate_caches(); os.chdir(REPO)

from bilinear_lmmd.data.preparation.prepare_coffee17 import discover_directory_samples
from bilinear_lmmd.data.preparation.audit_coffee17_provenance import audit_coffee17_provenance
from bilinear_lmmd.experiments.preprocessing_environment import verify_environment
verify_environment(PROJECT/'evidence/coffee17-preprocessing-runtime-v1/runtime_environment.json')

OOF=PROJECT/'oof/coffee17-preprocessing-primary-v1/merged'
ANALYSIS=PROJECT/'analysis/coffee17-preprocessing-primary-v1'
ANALYSIS.mkdir(parents=True,exist_ok=True)
MASTER=OOF/'primary_oof_table.csv'; SUMMARY=OOF/'primary_oof_summary.json'
if not MASTER.is_file() or not SUMMARY.is_file():
    raise FileNotFoundError('OOF merged evidence belum lengkap.')

subprocess.run([sys.executable,'-u','-m','bilinear_lmmd.experiments.run_preprocessing_bootstrap','--master-table',str(MASTER),'--output',str(ANALYSIS/'paired_bootstrap.json'),'--iterations','10000'],check=True)
subprocess.run([sys.executable,'-u','-m','bilinear_lmmd.experiments.run_preprocessing_analysis','--master-table',str(MASTER),'--output-dir',str(ANALYSIS)],check=True)

by_class=discover_directory_samples(INPUT)
ARCHIVE=WORK/'coffee17_original.zip'
if ARCHIVE.exists(): ARCHIVE.unlink()
with zipfile.ZipFile(ARCHIVE,'w',compression=zipfile.ZIP_STORED) as bundle:
    for class_name, paths in sorted(by_class.items()):
        for path in sorted(paths):
            info=zipfile.ZipInfo(f'{class_name}/{path.name}',date_time=(1980,1,1,0,0,0))
            info.compress_type=zipfile.ZIP_STORED; info.external_attr=0o644 << 16
            bundle.writestr(info,path.read_bytes())
CANONICAL=WORK/'coffee17_original_v1'; PROV=WORK/'coffee17_analysis_provenance'
for p in (CANONICAL,PROV):
    if p.exists(): shutil.rmtree(p)
audit_coffee17_provenance(ARCHIVE,PROV,canonical_root=CANONICAL)

subprocess.run([
    sys.executable,'-u','-m','bilinear_lmmd.experiments.run_preprocessing_efficiency',
    '--canonical-root',str(CANONICAL),
    '--clean-manifest',str(PROJECT/'evidence/coffee17-preprocessing-data-v1/clean_manifest.json'),
    '--output',str(ANALYSIS/'preprocessing_efficiency.json'),
    '--batch-sizes','1','16','--warmup','10','--iterations','50'
],check=True)
subprocess.run([
    sys.executable,'-u','-m','bilinear_lmmd.experiments.run_preprocessing_final_report',
    '--oof-summary',str(SUMMARY),
    '--bootstrap',str(ANALYSIS/'paired_bootstrap.json'),
    '--analysis-summary',str(ANALYSIS/'analysis_summary.json'),
    '--efficiency',str(ANALYSIS/'preprocessing_efficiency.json'),
    '--output-dir',str(ANALYSIS)
],check=True)
print('FINAL:',ANALYSIS/'FINAL_PREPROCESSING_REPORT.md')
for p in (REPO,CANONICAL,PROV):
    if p.exists(): shutil.rmtree(p,ignore_errors=True)
if ARCHIVE.exists(): ARCHIVE.unlink()
print('Klik Save Version untuk menyimpan report final sebagai Kaggle Notebook Output.')
